# SynthMorph 3D - Inference on Test Directory

In [ ]:
import math
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm

import sys
sys.path.insert(0, os.path.abspath('..'))

from synthmorph import configs as config
from synthmorph.loss import SoftDiceLoss
from synthmorph.network import SynthMorphUNet
from synthmorph.utils import create_integration_layer, PatchProcessor

In [ ]:
MODEL_PATH = "../outputs/train/best_model.pt"
TEST_DIR = ""  # set this to your test root dir (same structure as val_data_dir)
N_SAMPLES = 4
RANDOM_SEED = 42

IMAGE_FILENAME = config.val_image_filename
LABEL_FILENAME = config.val_label_filename
VAL_NUM_CLASSES = config.val_num_classes
IGNORE_LABEL = config.ignore_label
TARGET_SIZE = config.image_size

# Patch-based inference settings (match training config)
USE_PATCH_BASED = bool(getattr(config, "use_patch_based", False))
PATCH_SIZE = int(getattr(config, "patch_size", 64))
PATCH_OVERLAP = int(getattr(config, "patch_overlap", 16))

In [ ]:
import math
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm

import sys
sys.path.insert(0, os.path.abspath('..'))

from synthmorph import configs as config
from synthmorph.loss import SoftDiceLoss
from synthmorph.network import SynthMorphUNet
from synthmorph.utils import create_integration_layer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

dice_loss_fn = SoftDiceLoss(num_classes=VAL_NUM_CLASSES, ignore_label=IGNORE_LABEL)


def normalize_image(image: torch.Tensor) -> torch.Tensor:
    image = image.float()
    min_val = image.min()
    max_val = image.max()
    denom = torch.clamp(max_val - min_val, min=1e-6)
    return (image - min_val) / denom


def load_nifti_tensor(path: Path, is_label: bool, target_size: tuple[int, int, int], device: torch.device) -> torch.Tensor:
    volume_np = nib.load(str(path)).get_fdata()
    tensor = torch.from_numpy(volume_np)

    if is_label:
        tensor = tensor.long().unsqueeze(0).unsqueeze(0).float()
        resized = F.interpolate(tensor, size=target_size, mode="nearest")
        return resized.squeeze(0).squeeze(0).round().long().to(device)

    tensor = tensor.float().unsqueeze(0).unsqueeze(0)
    resized = F.interpolate(tensor, size=target_size, mode="trilinear", align_corners=True)
    resized = normalize_image(resized)
    return resized.squeeze(0).squeeze(0).to(device)


def field_to_sampling_grid(vector_field: torch.Tensor) -> torch.Tensor:
    batch_size, _, depth, height, width = vector_field.shape
    z = torch.linspace(-1.0, 1.0, depth, device=vector_field.device, dtype=vector_field.dtype)
    y = torch.linspace(-1.0, 1.0, height, device=vector_field.device, dtype=vector_field.dtype)
    x = torch.linspace(-1.0, 1.0, width, device=vector_field.device, dtype=vector_field.dtype)
    zz, yy, xx = torch.meshgrid(z, y, x, indexing="ij")
    base_grid = torch.stack((xx, yy, zz), dim=-1).unsqueeze(0).repeat(batch_size, 1, 1, 1, 1)

    norm = torch.tensor(
        [
            2.0 / max(width - 1, 1),
            2.0 / max(height - 1, 1),
            2.0 / max(depth - 1, 1),
        ],
        device=vector_field.device,
        dtype=vector_field.dtype,
    ).view(1, 3, 1, 1, 1)

    normalized_field = vector_field * norm
    return base_grid + normalized_field.permute(0, 2, 3, 4, 1)


def warp_label_map(label_map: torch.Tensor, integrated_field: torch.Tensor) -> torch.Tensor:
    if label_map.dim() == 4:
        label_map = label_map.unsqueeze(1)

    label_map = label_map.float()
    sampling_grid = field_to_sampling_grid(integrated_field)
    warped = F.grid_sample(
        label_map,
        sampling_grid,
        mode="nearest",
        padding_mode="border",
        align_corners=True,
    )
    return warped.squeeze(1).round().long()


def warp_intensity_map(image: torch.Tensor, integrated_field: torch.Tensor) -> torch.Tensor:
    if image.dim() == 4:
        image = image.unsqueeze(1)

    image = image.float()
    sampling_grid = field_to_sampling_grid(integrated_field)
    warped = F.grid_sample(
        image,
        sampling_grid,
        mode="bilinear",
        padding_mode="border",
        align_corners=True,
    )
    return warped.squeeze(1)


def discover_patients(root_dir: str) -> list[Path]:
    root = Path(root_dir)
    if not root_dir or not root.exists():
        return []

    patients = []
    for entry in root.iterdir():
        if not entry.is_dir():
            continue
        if (entry / IMAGE_FILENAME).exists() and (entry / LABEL_FILENAME).exists():
            patients.append(entry)
    return patients


def build_random_pairs(patients: list[Path]) -> list[tuple[Path, Path]]:
    if len(patients) < 2:
        return []

    shuffled = patients[:]
    random.shuffle(shuffled)
    num_pairs = math.ceil(len(shuffled) / 2)

    pairs = []
    pair_index = 0
    while len(pairs) < num_pairs:
        fixed_patient = shuffled[pair_index % len(shuffled)]
        moving_patient = shuffled[(pair_index + 1) % len(shuffled)]
        if fixed_patient != moving_patient:
            pairs.append((fixed_patient, moving_patient))
        pair_index += 2
    return pairs


def compute_dice(fixed_label: torch.Tensor, warped_label: torch.Tensor) -> float:
    return float((1.0 - dice_loss_fn(fixed_label, warped_label)).item())


def forward_patch_based_inference(
    model,
    fixed_image: torch.Tensor,
    moving_image: torch.Tensor,
    patch_processor: PatchProcessor,
    device: torch.device,
) -> torch.Tensor:
    """Inference with patch-based processing for memory efficiency."""
    model_input = torch.cat([fixed_image.unsqueeze(0), moving_image.unsqueeze(0)], dim=1)
    
    patches, positions = patch_processor.extract_patches(model_input[0])
    vector_field_patches = []
    
    for patch in patches:
        patch_batch = patch.unsqueeze(0)
        with torch.no_grad():
            patch_field = model(patch_batch)
        vector_field_patches.append(patch_field[0])
    
    full_vector_field = patch_processor.stitch_patches(
        vector_field_patches,
        positions,
        device=device,
        dtype=model_input.dtype,
    )
    return full_vector_field

In [ ]:
model = SynthMorphUNet().to(device)
checkpoint = torch.load(MODEL_PATH, map_location=device)
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)
model.eval()

integration_layer = create_integration_layer(
    image_size=TARGET_SIZE,
    integration_steps=config.integration_steps,
).to(device)

# Initialize patch processor if patch-based inference is enabled
patch_processor = None
if USE_PATCH_BASED:
    patch_processor = PatchProcessor(
        full_size=TARGET_SIZE,
        patch_size=PATCH_SIZE,
        overlap=PATCH_OVERLAP,
    )

print(f"Loaded model from: {MODEL_PATH}")
print(f"Running on: {device}")
if USE_PATCH_BASED:
    print(f"Using patch-based inference: {PATCH_SIZE}³ patches with {PATCH_OVERLAP} overlap")

In [ ]:
patients = discover_patients(TEST_DIR)
pairs = build_random_pairs(patients)

if len(pairs) == 0:
    raise ValueError(
        "No valid test pairs found. Set TEST_DIR and ensure each patient folder has "
        f"{IMAGE_FILENAME} and {LABEL_FILENAME}."
    )

sample_count = min(N_SAMPLES, len(pairs))
sample_indices = set(random.sample(range(len(pairs)), sample_count))

samples = []
dice_scores = []

model.eval()
with torch.no_grad():
    progress = tqdm(enumerate(pairs), total=len(pairs), desc="Testing")
    for idx, (fixed_patient, moving_patient) in progress:
        fixed_image = load_nifti_tensor(
            fixed_patient / IMAGE_FILENAME,
            is_label=False,
            target_size=TARGET_SIZE,
            device=device,
        )
        moving_image = load_nifti_tensor(
            moving_patient / IMAGE_FILENAME,
            is_label=False,
            target_size=TARGET_SIZE,
            device=device,
        )
        fixed_label = load_nifti_tensor(
            fixed_patient / LABEL_FILENAME,
            is_label=True,
            target_size=TARGET_SIZE,
            device=device,
        )
        moving_label = load_nifti_tensor(
            moving_patient / LABEL_FILENAME,
            is_label=True,
            target_size=TARGET_SIZE,
            device=device,
        )

        # Use patch-based or full-volume inference
        if USE_PATCH_BASED and patch_processor is not None:
            vector_field = forward_patch_based_inference(
                model,
                fixed_image,
                moving_image,
                patch_processor,
                device,
            )
        else:
            model_input = torch.stack([fixed_image, moving_image], dim=0).unsqueeze(0)
            vector_field = model(model_input)
        
        integrated_field = integration_layer(vector_field)

        warped_moving_label = warp_label_map(moving_label.unsqueeze(0), integrated_field)
        warped_moving_image = warp_intensity_map(moving_image.unsqueeze(0), integrated_field)

        dice_value = compute_dice(fixed_label.unsqueeze(0), warped_moving_label)
        dice_scores.append(dice_value)

        progress.set_postfix(dice=f"{dice_value:.4f}")

        if idx in sample_indices:
            samples.append(
                {
                    "fixed": fixed_image.cpu().numpy(),
                    "moving": moving_image.cpu().numpy(),
                    "warped": warped_moving_image.squeeze(0).cpu().numpy(),
                    "flow": integrated_field.squeeze(0).cpu().numpy(),
                    "dice": dice_value,
                    "fixed_name": fixed_patient.name,
                    "moving_name": moving_patient.name,
                }
            )

mean_dice = float(np.mean(dice_scores))
print(f"Patients found: {len(patients)}")
print(f"Pairs evaluated: {len(pairs)}")
print(f"Average Dice on test set: {mean_dice:.4f}")

## Random Sample Visualizations and Flow Field

In [ ]:
def norm(vol: np.ndarray) -> np.ndarray:
    lo, hi = vol.min(), vol.max()
    return (vol - lo) / (hi - lo + 1e-6)


if len(samples) == 0:
    print("No samples were captured. Increase N_SAMPLES or check TEST_DIR.")
else:
    rows = len(samples)
    fig, axes = plt.subplots(rows, 5, figsize=(16, 3.5 * rows))
    if rows == 1:
        axes = np.expand_dims(axes, axis=0)

    col_titles = [
        "Fixed",
        "Moving",
        "Warped Moving",
        "Overlay (fixed=green, warped=magenta)",
        "Flow Magnitude + In-plane Vectors",
    ]
    for col, title in enumerate(col_titles):
        axes[0, col].set_title(title, fontsize=10)

    for row, sample in enumerate(samples):
        fixed = sample["fixed"]
        moving = sample["moving"]
        warped = sample["warped"]
        flow = sample["flow"]
        score = sample["dice"]

        mid = fixed.shape[0] // 2
        f2d = norm(fixed[mid])
        m2d = norm(moving[mid])
        w2d = norm(warped[mid])

        h, w = f2d.shape
        overlay = np.zeros((h, w, 3), dtype=np.float32)
        overlay[:, :, 1] = f2d
        overlay[:, :, 0] = w2d
        overlay[:, :, 2] = w2d

        flow_x = flow[0, mid]
        flow_y = flow[1, mid]
        flow_z = flow[2, mid]
        flow_mag = np.sqrt(flow_x**2 + flow_y**2 + flow_z**2)

        axes[row, 0].imshow(f2d, cmap="gray", vmin=0, vmax=1)
        axes[row, 1].imshow(m2d, cmap="gray", vmin=0, vmax=1)
        axes[row, 2].imshow(w2d, cmap="gray", vmin=0, vmax=1)
        axes[row, 3].imshow(overlay, vmin=0, vmax=1)
        axes[row, 4].imshow(flow_mag, cmap="inferno")

        step = max(1, min(h, w) // 24)
        yy, xx = np.mgrid[0:h:step, 0:w:step]
        axes[row, 4].quiver(
            xx,
            yy,
            flow_x[::step, ::step],
            flow_y[::step, ::step],
            color="cyan",
            angles="xy",
            scale_units="xy",
            scale=2.0,
            alpha=0.8,
            width=0.002,
        )

        axes[row, 0].set_ylabel(
            f"Dice={score:.3f}\\n{sample['fixed_name']} <- {sample['moving_name']}",
            fontsize=9,
        )

        for ax in axes[row]:
            ax.axis("off")

    plt.suptitle(
        "SynthMorph Test Inference: Random Samples and Integrated Flow Field",
        fontsize=13,
        y=1.02,
    )
    plt.tight_layout()
    plt.show()